<h1 align="center" style="color: #1f77b4; font-weight: bold;">
    Проект: ETL и подготовка данных процессов ферментации (Инсулин / Гларгин)
</h1>

<p align="center" style="font-size: 16px; color: #555555;">
    <b>Цель:</b> Очистка и нормализация временных рядов ферментации, выявление аномалий датчиков, агрегация технологических параметров на уровне производственных серий и объединение с лабораторными KPI для последующего анализа и визуализации в Power BI.
</p>

---
## 1. Инициализация и загрузка исходных данных

In [1]:
import numpy as np
import pandas as pd

# Загрузка исходного файла с двухуровневой шапкой
df = pd.read_excel('Биосинтез инсулина_02.xlsx', header=[0, 1])

# Проверка успешности загрузки и структуры таблицы
df.head()

№ Серии                Тип   Дата ферментации  \
  Unnamed: 0_level_1 Unnamed: 1_level_1 Unnamed: 2_level_1   
0     BATCH-2024-001            Инсулин         2024-05-12   
1                NaN                NaN                NaT   
2                NaN                NaN                NaT   
3                NaN                NaN                NaT   
4                NaN                NaN                NaT   

             Ферментер   время роста, час время синтеза, час  \
    Unnamed: 3_level_1 Unnamed: 4_level_1 Unnamed: 5_level_1   
0  Посевной реактор №1                0.0                NaN   
1                  NaN                0.5                NaN   
2                  NaN                1.0                NaN   
3                  NaN                1.5                NaN   
4                  NaN                2.0                NaN   

           N, об\мин          V, м3\час                 рН              рО, %  \
  Unnamed: 6_level_1 Unnamed: 7_level_1 Unnamed: 8_level_1 Unnamed: 9_level_1   
0              100.0                1.0               7.54               40.6   
1              100.0                1.0               7.51               18.9   
2              116.0                1.7               7.46               26.5   
3              152.0                3.2               7.40               29.8   
4              189.0                4.2               7.31               28.9   

               Р, бар          Осмос, атм                  ОП Глюкоза, л  \
  Unnamed: 10_level_1 Unnamed: 11_level_1 Unnamed: 12_level_1       Доза   
0                0.12                18.5                0.83        0.2   
1                0.30                18.9                0.90        0.0   
2                0.31                19.1                1.01        0.0   
3                0.31                19.2                1.49        0.0   
4                0.37                19.2                2.43        0.0   

        Подпитка, л         щелочь\кислота, л          Примечание  
  Сумма        Доза Сумма Unnamed: 17_level_1 Unnamed: 18_level_1  
0   0.2         NaN   NaN                 NaN                 NaN  
1   0.2         NaN   NaN                 NaN                 NaN  
2   0.2         NaN   NaN                 NaN                 NaN  
3   0.2         NaN   NaN                 NaN                 NaN  
4   0.2         NaN   NaN                 NaN                 NaN

## 2. Нормализация наименований колонок
> Очистка мульти-индекса Excel: сжатие двухуровневой шапки в одномерный список названий, удаление служебных меток `Unnamed` и замена спецсимволов.

In [2]:
# Формирование чистых наименований столбцов
new_columns = []

for col in df.columns:
    level_1 = col[0].strip()
    level_2 = col[1].strip()

    if 'Unnamed' in level_2:
        new_columns.append(level_1)
    else:
        clean_1 = level_1.replace('\\', ' ')
        new_columns.append(f'{clean_1}_{level_2}')

df.columns = new_columns

# Проверяем обновленные названия колонок
df.head()

,№ Серии,Тип,Дата ферментации,Ферментер,"время роста, час","время синтеза, час","N, об\мин","V, м3\час",рН,"рО, %","Р, бар","Осмос, атм",ОП,"Глюкоза, л_Доза","Глюкоза, л_Сумма","Подпитка, л_Доза","Подпитка, л_Сумма","щелочь\кислота, л",Примечание
0,BATCH-2024-001,Инсулин,2024-05-12,Посевной реактор №1,0.0,NaN,100.0,1.0,7.54,40.6,0.12,18.5,0.83,0.2,0.2,NaN,NaN,NaN,NaN
1,NaN,NaN,NaT,NaN,0.5,NaN,100.0,1.0,7.51,18.9,0.30,18.9,0.90,0.0,0.2,NaN,NaN,NaN,NaN
2,NaN,NaN,NaT,NaN,1.0,NaN,116.0,1.7,7.46,26.5,0.31,19.1,1.01,0.0,0.2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaT,NaN,1.5,NaN,152.0,3.2,7.40,29.8,0.31,19.2,1.49,0.0,0.2,NaN,NaN,NaN,NaN
4,NaN,NaN,NaT,NaN,2.0,NaN,189.0,4.2,7.31,28.9,0.37,19.2,2.43,0.0,0.2,NaN,NaN,NaN,NaN


## 3. Обработка пропущенных значений
> Данные датчиков содержат пропуски двух типов:
> 1. **Статические параметры процесса** (pH, ОП, давление, обороты, дата) — заполняются методом `ffill()` (протягивание последнего известного значения).
> 2. **Дискретные дозировки** (подпитка, дозы глюкозы, щелочь/кислота) — отсутствующие значения означают отсутствие подачи в данный момент времени, заполняются `0`.

In [3]:
# Заполнение непрерывных показаний (Forward Fill)
cols_to_ffill = ['№ Серии', 'Дата ферментации', 'Тип', 'Ферментер', 'N, об\мин', 'V, м3\час', 'Глюкоза, л_Сумма', 'рН', 'Р, бар', 'ОП']

df[cols_to_ffill] = df[cols_to_ffill].ffill() 

# Заполнение дискретных подач нулями
cols_to_zero = ['время синтеза, час', 'Подпитка, л_Доза', 'Подпитка, л_Сумма', 'щелочь\кислота, л', 'Глюкоза, л_Доза']

df[cols_to_zero] = df[cols_to_zero].fillna(0)

# Проверка ликвидации пропусков
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1221 entries, 0 to 1220
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   № Серии             1221 non-null   object        
 1   Тип                 1221 non-null   object        
 2   Дата ферментации    1221 non-null   datetime64[ns]
 3   Ферментер           1221 non-null   object        
 4   время роста, час    1221 non-null   float64       
 5   время синтеза, час  1221 non-null   float64       
 6   N, об\мин           1221 non-null   float64       
 7   V, м3\час           1221 non-null   float64       
 8   рН                  1221 non-null   float64       
 9   рО, %               1106 non-null   float64       
 10  Р, бар              1221 non-null   float64       
 11  Осмос, атм          699 non-null    float64       
 12  ОП                  1221 non-null   float64       
 13  Глюкоза, л_Доза     1221 non-null   float64     

## 4. Обнаружение и фильтрация технологических аномалий
> Валидация физических диапазонов датчиков. Значения, выходящие за пределы технологического регламента (pH > 14, расход воздуха V > 500 м³/час, давление P > 2 бар), идентифицируются как сбои оборудования, заменяются на `NaN` и восстанавливаются корректными показаниями методом `ffill()`.

In [4]:
# 1. Поиск и вывод аномальных записей для контроля
anomalies = df[(df['рН'] > 14) | (df['V, м3\час'] > 500) | (df['Р, бар'] > 2.0)]

print(f'Обнаружено аномальных записей датчиков: {len(anomalies)}')

anomalies[['№ Серии', 'время роста, час', 'рН', 'V, м3\час', 'Р, бар', 'Примечание']]

Обнаружено аномальных записей датчиков: 3


,№ Серии,"время роста, час",рН,"V, м3\час","Р, бар",Примечание
667,BATCH-2024-018,1.5,70.60,108.70,0.33,NaN
758,BATCH-2024-020,7.5,7.01,1432.90,0.44,NaN
1115,BATCH-2024-029,10.0,31.20,6.98,15.60,NaN


In [5]:
# 2. Замена сбойных значений датчиков на NaN
df.loc[df['рН']>14, 'рН'] = np.nan
df.loc[df['V, м3\час'] > 500, 'V, м3\час'] = np.nan
df.loc[df['Р, бар'] > 3, 'Р, бар'] = np.nan

# 3. Восстановление корректных значений из предыдущих замеров
df[['рН', 'V, м3\час', 'Р, бар']] = df[['рН', 'V, м3\час', 'Р, бар']].ffill()

# 4. Проверка максимальных значений после очистки
df[['рН', 'V, м3\час', 'Р, бар']].max()

рН             7.54
V, м3\час    359.60
Р, бар         1.40
dtype: float64

## 5. Корректировка времени синтеза и агрегация по сериям
> Исправление выбросов в продолжительности синтеза (> 25 часов) с последующей свёрткой поминутной телеметрии в уровень серий (`Fact Batch Level`). 
> Вычисляются ключевые агрегаты: максимальная оптическая плотность (макс_ОП), профиль pH, суммарный расход сырья и длительность фаз.

In [6]:
# Фильтрация выбросов во времени синтеза (> 25 часов) и восстановление
df.loc[df['время синтеза, час']>25, 'время синтеза, час'] = np.nan
df['время синтеза, час'] = df.groupby('№ Серии')['время синтеза, час'].ffill()

df['время синтеза, час'].max()

18.5

In [7]:
# Короткий номер серии для отображения в Power BI
df['Номер серии'] = (
    df['№ Серии']
    .astype('string')
    .str.extract(r'-(\d+)$', expand=False)
)

# Сохраняем детальные данные ферментации для Power BI
df_detail = df.copy()

# Экспорт временных рядов
df_detail.to_csv(
    'fermentation_timeseries.csv',
    index=False,
    encoding='utf-8-sig'
)

print(f'Экспортировано записей: {len(df_detail)}')
print(f'Количество столбцов: {len(df_detail.columns)}')

Экспортировано записей: 1221
Количество столбцов: 20


In [8]:
# Агрегация до уровня отдельных серий

batch_summary = df.groupby(['№ Серии', 'Тип'], as_index=False).agg(
    Дата_старта=('Дата ферментации', 'min'),
    Макс_ОП=('ОП', 'max'),
    Длительность_роста=('время роста, час', 'max'),
    Длительность_синтеза=('время синтеза, час', 'max'),
    Средний_pH=('рН', 'mean'),
    Мин_pH=('рН', 'min'),
    Макс_pH=('рН', 'max'),
    Всего_глюкозы_л=('Глюкоза, л_Доза', 'sum'),
    Всего_подпитки_л=('Подпитка, л_Доза', 'sum')
)

batch_summary['Номер серии'] = (
    batch_summary['№ Серии']
    .astype('string')
    .str.extract(r'-(\d+)$', expand=False)
)

# Проверка итогового агрегированного датасета
batch_summary.head()

,№ Серии,Тип,Дата_старта,Макс_ОП,Длительность_роста,Длительность_синтеза,Средний_pH,Мин_pH,Макс_pH,Всего_глюкозы_л,Всего_подпитки_л,Номер серии
0,BATCH-2024-001,Инсулин,2024-05-12,167.4,14.60,12.0,7.166341,6.96,7.54,374.25,288.8,001
1,BATCH-2024-002,Инсулин,2024-05-13,154.4,12.15,10.0,7.011111,6.87,7.43,292.22,288.4,002
2,BATCH-2024-003,Инсулин,2024-05-19,170.6,14.25,12.0,6.957000,6.77,7.23,379.42,288.3,003
3,BATCH-2024-004,Инсулин,2024-05-21,184.2,13.75,12.0,7.058718,6.92,7.47,298.70,294.3,004
4,BATCH-2024-005,Инсулин,2024-05-23,132.1,14.00,12.0,7.047750,6.99,7.34,340.35,289.5,005


## 6. Объединение с лабораторными KPI.
> Подтягивание данных лабораторного контроля качества (микроскопия) по ключу `№ Серии`. 
> Обработка пропущенных значений в показателях лизиса/ТВ (заполнение `0`) и удаление служебных колонок.

In [9]:
# Загрузка лабораторного журнала
df_summary = pd.read_excel('Сводные_значения_биосинтеза.xlsx')

# Объединение с агрегированной телеметрией (Inner Join по № Серии)
df_merge = pd.merge(batch_summary, df_summary, on='№ Серии', how='inner')

# Удаление технического индекса Excel
df_merge.drop(columns='№ по порядку', inplace=True, errors='ignore')

# Заполнение пропусков в лабораторной микроскопии нулями
lab_cols = ['% Лизированных клеток', '% "свободных" ТВ', '% клеток без ТВ']
df_merge[lab_cols] = df_merge[lab_cols].fillna(0)

# Проверка готовности структуры финального датасета
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   № Серии                32 non-null     object        
 1   Тип                    32 non-null     object        
 2   Дата_старта            32 non-null     datetime64[ns]
 3   Макс_ОП                32 non-null     float64       
 4   Длительность_роста     32 non-null     float64       
 5   Длительность_синтеза   32 non-null     float64       
 6   Средний_pH             32 non-null     float64       
 7   Мин_pH                 32 non-null     float64       
 8   Макс_pH                32 non-null     float64       
 9   Всего_глюкозы_л        32 non-null     float64       
 10  Всего_подпитки_л       32 non-null     float64       
 11  Номер серии            32 non-null     string        
 12  % Лизированных клеток  32 non-null     float64       
 13  % "своб

## 7. Экспорт итогового датасета для Power BI
> Сохранение валидированной сводной таблицы 32 серий в CSV.

In [10]:
df_merge.to_csv('batch_summary_final.csv', index=False, encoding='utf-8-sig')